In [2]:
import pandas as pd
import numpy as np
import holidays
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, f1_score, confusion_matrix, classification_report)

# True : model predicts the CHANGE (tomorrow - today), then today's AQI is added back
# False: model predicts tomorrow's AQI level directly
PREDICT_CHANGE = True
# True : today's pollutant readings are used as features
# False: AQI history + calendar only
USE_POLLUTANTS = True

POLLUTANTS = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3',
              'Benzene', 'Toluene', 'Xylene']

# ---------------------------------------------------------------
# PART 1: YOUR CLEANING, adapted to city_day.csv
# (city_day already has ONE ROW PER CITY PER DAY
#  Pollutant columns are KEPT; only AQI_Bucket is dropped because it is derived from AQI.)
# ---------------------------------------------------------------
df = pd.read_csv('AQIData2015to20/city_day.csv')
totalrows = len(df)
emptyrows = df['AQI'].isnull().sum()
print(totalrows, ' number of rows with ', emptyrows, ' empty')

df = df.dropna(subset=['AQI', 'City', 'Date'])
df = df.drop_duplicates()
df['Date'] = pd.to_datetime(df['Date'])
df = df.drop(columns=['AQI_Bucket'])
df[POLLUTANTS] = df[POLLUTANTS].mask(df[POLLUTANTS] < 0)   # negative readings are sensor errors
print('Dataset size after cleaning ', len(df))

years = df['Date'].dt.year.unique()
india_holidays = holidays.India(years=years)

# ---------------------------------------------------------------
# PART 2: REMOVE IMPOSSIBLE AQI (the scale tops out at 500)
# ---------------------------------------------------------------
print((df['AQI'] > 500).sum(), "rows with AQI above 500 removed")
daily = df[df['AQI'] <= 500].copy()
daily['date'] = daily['Date'].dt.normalize()
daily = daily.groupby(['City', 'date'], as_index=False)[['AQI'] + POLLUTANTS].mean()   # one row per city-day


# ---------------------------------------------------------------
# PART 3: TARGET (tomorrow's AQI) + LAG FEATURES (recent AQI), per city
# ---------------------------------------------------------------
def build(g):
    city = g['City'].iloc[0]
    g = g.set_index('date').asfreq('D')        # missing days become NaN, so shifts mean real days
    g['City'] = city
    g['AQI_next'] = g['AQI'].shift(-1)         # TARGET: tomorrow
    g['lag1'] = g['AQI'].shift(1)              # yesterday
    g['lag2'] = g['AQI'].shift(2)              # 2 days ago
    g['lag7'] = g['AQI'].shift(7)              # a week ago
    g['roll7'] = g['AQI'].rolling(7, min_periods=1).mean()   # last 7 days incl. today
    g['roll3'] = g['AQI'].rolling(3, min_periods=1).mean()
    g['roll7_std'] = g['AQI'].rolling(7, min_periods=2).std()
    g['date_next'] = g.index + pd.Timedelta(days=1)          # the day being predicted
    return g.reset_index()


daily = pd.concat([build(g) for _, g in daily.groupby('City')], ignore_index=True)

# calendar features of the day being predicted (known in advance)
dn = daily['date_next']
daily['month_sin'] = np.sin(2 * np.pi * dn.dt.month / 12)
daily['month_cos'] = np.cos(2 * np.pi * dn.dt.month / 12)
daily['Weekday'] = dn.dt.dayofweek
daily['is_weekend'] = daily['Weekday'].isin([5, 6]).astype(int)
daily['is_festival'] = dn.dt.date.apply(lambda x: int(x in india_holidays))

# within 3 days of Diwali (pollution spikes around it)
diwali_days = np.array([np.datetime64(day, 'D') for day, name in india_holidays.items()
                        if 'diwali' in name.lower() or 'deepavali' in name.lower()],
                       dtype='datetime64[D]')
print("Diwali dates found in holidays library:", len(diwali_days), "(should be about 6)")
dn_days = dn.values.astype('datetime64[D]')
daily['near_diwali'] = (np.abs(dn_days[:, None] - diwali_days[None, :])
                        <= np.timedelta64(3, 'D')).any(axis=1).astype(int)

# keep rows where today and tomorrow are both known; fill small gaps in the lags
daily = daily.dropna(subset=['AQI', 'AQI_next'])
daily['lag1'] = daily['lag1'].fillna(daily['AQI'])
daily['lag2'] = daily['lag2'].fillna(daily['lag1'])
daily['lag7'] = daily['lag7'].fillna(daily['roll7'])
daily['trend'] = daily['AQI'] - daily['lag1']        # rising or falling today?
daily['roll7_std'] = daily['roll7_std'].fillna(0)

# AQI categories for tomorrow (target) and today (for the baseline)
BINS = [-np.inf, 50, 100, 200, 300, 400, np.inf]     # CPCB category limits
ORDER = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']
daily['y_cls'] = pd.cut(daily['AQI_next'], bins=BINS, labels=False).astype(int)
daily['y_today'] = pd.cut(daily['AQI'], bins=BINS, labels=False).astype(int)

# ---------------------------------------------------------------
# PART 4: CHRONOLOGICAL SPLIT (by the date being predicted)
# ---------------------------------------------------------------
train = daily[daily['date_next'] < '2019-01-01']
val = daily[(daily['date_next'] >= '2019-01-01') & (daily['date_next'] < '2020-01-01')]
test = daily[daily['date_next'] >= '2020-01-01']
trainval = pd.concat([train, val])
print(f"train {len(train)} | val {len(val)} | test {len(test)}")

# pollutants with more than 40% missing in TRAINING data are dropped (decided on train only)
miss_pct = train[POLLUTANTS].isna().mean() * 100
print("\nPollutant missing % in training data:\n", miss_pct.round(1).to_string())
POLL_FEATS = [c for c in POLLUTANTS if miss_pct[c] <= 40] if USE_POLLUTANTS else []
print("Pollutants used as features:", POLL_FEATS)

base_features = ['AQI', 'lag1', 'lag2', 'lag7', 'roll7',
                 'month_sin', 'month_cos', 'Weekday', 'is_weekend', 'is_festival']
extra_features = ['trend', 'roll3', 'roll7_std', 'near_diwali']
features = base_features + extra_features + POLL_FEATS
cities = sorted(daily['City'].unique())


def make_prep(fit_df):
    """Learn pollutant fill-in values (medians) from fit_df ONLY, return a feature builder."""
    city_med = fit_df.groupby('City')[POLL_FEATS].median()
    glob_med = fit_df[POLL_FEATS].median()

    def make_X(d):
        d = d.copy()
        for c in POLL_FEATS:
            d[c] = d[c].fillna(d['City'].map(city_med[c])).fillna(glob_med[c])
        X = d[features].reset_index(drop=True)
        dummies = pd.get_dummies(pd.Categorical(d['City'], categories=cities), prefix='City').astype(float)
        return pd.concat([X, dummies], axis=1)
    return make_X


def metrics(y, p):
    return dict(MAE=mean_absolute_error(y, p),
                RMSE=np.sqrt(mean_squared_error(y, p)),
                R2=r2_score(y, p))


def target(d):
    return d['AQI_next'] - d['AQI'] if PREDICT_CHANGE else d['AQI_next']


def to_level(d, p):
    return p + d['AQI'].values if PREDICT_CHANGE else p


# ---------------------------------------------------------------
# PART 5: CHOOSE SETTINGS ON VALIDATION ONLY
# ---------------------------------------------------------------
prep_tr = make_prep(train)                 # fill-in values from training data only
X_tr, X_val = prep_tr(train), prep_tr(val)
print("\nPersistence on validation:", {k: round(float(v), 3) for k, v in metrics(val['AQI_next'], val['AQI']).items()})

results = []
for leaf in [3, 5, 10, 20]:
    for depth in [10, 20, None]:
        m = RandomForestRegressor(n_estimators=200, min_samples_leaf=leaf, max_depth=depth,
                                  n_jobs=-1, random_state=42)
        m.fit(X_tr, target(train))
        p = to_level(val, m.predict(X_val))
        results.append({'leaf': leaf, 'depth': depth, **metrics(val['AQI_next'], p)})
res = pd.DataFrame(results).sort_values('RMSE')
print(res.round(3).head())

best = res.iloc[0]
best_depth = None if pd.isna(best['depth']) else int(best['depth'])

# ---------------------------------------------------------------
# PART 6: REFIT ON TRAIN+VAL, SCORE ON 2020 TEST ONCE
# ---------------------------------------------------------------
prep_tv = make_prep(trainval)              # fill-in values from train+val only
Xtv, X_te = prep_tv(trainval), prep_tv(test)

rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=int(best['leaf']),
                           max_depth=best_depth, n_jobs=-1, random_state=42)
rf.fit(Xtv, target(trainval))
pred = to_level(test, rf.predict(X_te))

print("\nTEST (2020)  | USE_POLLUTANTS =", USE_POLLUTANTS, "| PREDICT_CHANGE =", PREDICT_CHANGE)
print("Persistence  :", {k: round(float(v), 3) for k, v in metrics(test['AQI_next'], test['AQI']).items()})
print("Random Forest:", {k: round(float(v), 3) for k, v in metrics(test['AQI_next'], pred).items()})
print("Bias  persistence:", round(float((test['AQI'] - test['AQI_next']).mean()), 2),
      "| forest:", round(float((pred - test['AQI_next']).mean()), 2))

# ---------------------------------------------------------------
# PART 7: PER-CITY TABLE (forest vs persistence) AND FEATURE IMPORTANCE
# ---------------------------------------------------------------
out = test.assign(pred=pred.round(1))
pc = out.groupby('City')[['AQI_next', 'pred', 'AQI']].apply(lambda g: pd.Series({
    'RF_MAE': mean_absolute_error(g['AQI_next'], g['pred']),
    'Persist_MAE': mean_absolute_error(g['AQI_next'], g['AQI']),
    'mean_actual': g['AQI_next'].mean(),
    'RF_bias': g['pred'].mean() - g['AQI_next'].mean(),
    'n': len(g)})).round(1)
pc['gain_%'] = ((1 - pc['RF_MAE'] / pc['Persist_MAE']) * 100).round(1)
print(pc.sort_values('gain_%'))

imp = pd.Series(rf.feature_importances_, index=X_te.columns).sort_values(ascending=False)
print(imp.head(10))

# ---------------------------------------------------------------
# PART 8: PREDICT TOMORROW'S AQI CATEGORY
# ---------------------------------------------------------------
def cls_metrics(y, p):
    return dict(Accuracy=accuracy_score(y, p),
                MacroF1=f1_score(y, p, average='macro', zero_division=0),
                Within1=float((np.abs(np.asarray(y) - np.asarray(p)) <= 1).mean()))


best_leaf, best_f1 = 5, -1
for leaf in [3, 5, 10, 20]:
    c = RandomForestClassifier(n_estimators=200, min_samples_leaf=leaf,
                               class_weight='balanced_subsample', n_jobs=-1, random_state=42)
    c.fit(X_tr, train['y_cls'])
    f1 = cls_metrics(val['y_cls'], c.predict(X_val))['MacroF1']
    if f1 > best_f1:
        best_leaf, best_f1 = leaf, f1

clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=best_leaf,
                             class_weight='balanced_subsample', n_jobs=-1, random_state=42)
clf.fit(Xtv, trainval['y_cls'])
cls_pred = clf.predict(X_te)

# same classifier WITHOUT class weights, to see the accuracy / rare-class trade-off
clf_nw = RandomForestClassifier(n_estimators=300, min_samples_leaf=best_leaf,
                                n_jobs=-1, random_state=42).fit(Xtv, trainval['y_cls'])
cls_pred_nw = clf_nw.predict(X_te)

# second route: use the regression forecast and convert it to a category
binned = np.digitize(pred, [50, 100, 200, 300, 400], right=True)

table = pd.DataFrame({
    'Persistence (today bucket)': cls_metrics(test['y_cls'], test['y_today']),
    'RF classifier (weighted)': cls_metrics(test['y_cls'], cls_pred),
    'RF classifier (unweighted)': cls_metrics(test['y_cls'], cls_pred_nw),
    'RF regressor -> bucket': cls_metrics(test['y_cls'], binned)}).T.round(3)
print("\nBUCKET RESULTS (TEST 2020)")
print(table)

cm = confusion_matrix(test['y_cls'], cls_pred, labels=range(6))
print(pd.DataFrame(cm, index=[f"actual {o}" for o in ORDER], columns=[f"pred {o}" for o in ORDER]))
print(classification_report(test['y_cls'], cls_pred, labels=range(6),
                            target_names=ORDER, zero_division=0))


29531  number of rows with  4681  empty
Dataset size after cleaning  24850
543 rows with AQI above 500 removed


/tmp/ipykernel_3956/3070780685.py:71: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  daily['Weekday'] = dn.dt.dayofweek


Diwali dates found in holidays library: 6 (should be about 6)
train 12543 | val 6741 | test 4330

Pollutant missing % in training data:
 PM2.5       3.4
PM10       41.1
NO          1.3
NO2         1.0
NOx         9.8
NH3        33.7
CO          1.0
SO2         2.4
O3          3.2
Benzene    15.2
Toluene    19.2
Xylene     64.9
Pollutants used as features: ['PM2.5', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene']

Persistence on validation: {'MAE': 23.93, 'RMSE': 38.735, 'R2': 0.825}
   leaf  depth     MAE    RMSE     R2
5     5    NaN  20.163  31.276  0.886
4     5   20.0  20.153  31.282  0.886
7    10   20.0  20.103  31.298  0.886
8    10    NaN  20.107  31.299  0.886
1     3   20.0  20.246  31.360  0.885

TEST (2020)  | USE_POLLUTANTS = True | PREDICT_CHANGE = True
Persistence  : {'MAE': 17.695, 'RMSE': 30.45, 'R2': 0.805}
Random Forest: {'MAE': 16.116, 'RMSE': 25.047, 'R2': 0.868}
Bias  persistence: 0.9 | forest: 4.04
                    RF_MAE  Persist_MAE  mean